# 03b — Ekstraksi Fitur: Contextual/Transformer Embeddings

**Tugas 3, Butir 1a.ii**: Mengekstraksi representasi teks menggunakan model transformer pre-trained yang sudah di-fine-tune pada domain keuangan.

Notebook ini menghasilkan embedding kontekstual dari artikel berita menggunakan **FinBERT** (model BERT yang di-fine-tune pada teks keuangan) dan **DeBERTa-v3-small** (model general-purpose sebagai perbandingan).

### Perbedaan dari fitur sebelumnya (notebook 03)

| Aspek | TF-IDF/SVD (notebook 03) | Transformer Embeddings (notebook ini) |
|---|---|---|
| **Tipe** | Static, bag-of-words | Contextual, token-level |
| **Konteks** | Tidak memperhitungkan urutan kata | Memperhitungkan konteks kalimat penuh |
| **Pre-training** | Tidak ada | Dilatih pada jutaan dokumen (general + keuangan) |
| **Dimensi** | 50 (setelah SVD) | 768 (FinBERT) / 384 (DeBERTa-small) |

### Pipeline

1. Load model pre-trained (FinBERT / DeBERTa-v3-small)
2. Tokenisasi teks artikel (judul + isi, dipotong menjadi 512 token)
3. Inference → ambil hidden state layer terakhir
4. Pooling → **mean pooling** atas token non-padding (lebih stabil dari `[CLS]`)
5. Agregasi harian → rata-rata embedding semua artikel di hari yang sama
6. Reduksi dimensi → PCA (fit pada train) ke 50 komponen
7. Gabung dengan dataset kurs → simpan sebagai `model_dataset_transformer.csv`

### Input & Output

**Input:**
- `data/split/articles_split.csv` (artikel per trading day)
- `data/split/daily_kurs_split.csv` (kurs harian)
- `data/feature_extracted/features_lexicon.csv` (fitur VADER/LM dari notebook 03)

**Output (semua di `data/feature_extracted/`):**
- `embeddings_finbert_article.npz` — embedding per artikel (raw, sebelum agregasi)
- `embeddings_finbert_daily.csv` — embedding harian setelah agregasi + PCA
- `model_dataset_transformer.csv` — dataset siap modeling (kurs + lag + lexicon + transformer embeddings)

### Catatan leakage

Model transformer **tidak dilatih** dari data (hanya inference), sehingga tidak ada risiko data leakage. Yang di-fit hanya pada train: PCA untuk reduksi dimensi.

## 0. Setup & Install

In [ ]:
# Pastikan library yang dibutuhkan sudah terinstall
# !pip install transformers accelerate torch

import warnings
warnings.filterwarnings("ignore")

import gc
import json
import re
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm

# === Tentukan root repository ===
def find_repo_root():
    """Cari folder root repo (yang berisi data/ dan src/)."""
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "data").is_dir() and (d / "src").is_dir():
            return d
    raise FileNotFoundError(f"Root repo (berisi data/ dan src/) tidak ditemukan dari {here}")

ROOT = find_repo_root()
SPLIT_DIR = ROOT / "data" / "split"
FEATURE_DIR = ROOT / "data" / "feature_extracted"
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

# === Device ===
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Repository : {ROOT}")
print(f"Device     : {DEVICE}")
if torch.cuda.is_available():
    print(f"GPU        : {torch.cuda.get_device_name(0)}")
    print(f"VRAM       : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Konfigurasi

In [ ]:
# =====================================================================
# KONFIGURASI UTAMA — ubah di sini untuk eksperimen
# =====================================================================

# Model transformer yang digunakan.
# Opsi yang sudah diuji:
#   - "ProsusAI/finbert"                → FinBERT (domain keuangan, 768-d)
#   - "microsoft/deberta-v3-small"      → DeBERTa-v3-small (general, 384-d)
#   - "yiyanghkust/finbert-tone"        → FinBERT-Tone (sentimen keuangan, 768-d)
MODEL_NAME = "ProsusAI/finbert"

# Panjang maksimum token (BERT-family = 512)
MAX_LENGTH = 512

# Batch size untuk inference (sesuaikan dengan VRAM)
# RTX 3050 6GB → batch_size=8 aman untuk FinBERT 512 token
BATCH_SIZE = 8

# Teks yang di-embed: "title", "body", atau "both" (judul + isi)
EMBED_FIELD = "both"

# Jumlah komponen PCA untuk reduksi dimensi (fit hanya pada train)
PCA_COMPONENTS = 50

# Timing berita (harus sama dengan notebook 03)
NEWS_TIMING = "sebelum_penetapan_besok"

# Label singkat untuk penamaan file (otomatis dari nama model)
MODEL_SHORT = MODEL_NAME.split("/")[-1].replace("-", "_")

print(f"Model           : {MODEL_NAME}")
print(f"Max token       : {MAX_LENGTH}")
print(f"Batch size      : {BATCH_SIZE}")
print(f"Embed field     : {EMBED_FIELD}")
print(f"PCA components  : {PCA_COMPONENTS}")
print(f"News timing     : {NEWS_TIMING}")

## 2. Muat data artikel dan kurs

In [ ]:
daily = pd.read_csv(SPLIT_DIR / "daily_kurs_split.csv", parse_dates=["date"])
arts  = pd.read_csv(SPLIT_DIR / "articles_split.csv", parse_dates=["trading_date"])

# Isi kolom teks kosong dengan string kosong
for col in ["title_clean", "body_clean", "title_bow", "body_bow"]:
    if col in arts.columns:
        arts[col] = arts[col].fillna("")

print(f"Artikel : {len(arts):,} ({arts['split'].value_counts().to_dict()})")
print(f"Hari bursa: {len(daily):,}")
print(f"Kolom artikel: {list(arts.columns)}")

## 3. Siapkan teks untuk embedding

Untuk contextual embedding, kita gunakan **teks asli** (`title_clean` dan `body_clean`), bukan BoW.
Ini karena transformer memerlukan urutan kata dan tanda baca yang utuh untuk memahami konteks.

Teks judul dan isi digabung dengan separator `[SEP]` agar model tahu batas antara keduanya.

In [ ]:
def prepare_text(row):
    """Gabungkan teks judul dan/atau isi untuk embedding."""
    title = str(row.get("title_clean", "")).strip()
    body  = str(row.get("body_clean", "")).strip()
    
    if EMBED_FIELD == "title":
        return title
    elif EMBED_FIELD == "body":
        return body
    else:  # "both"
        # Gabung judul + isi; tokenizer akan menangani truncation
        if title and body:
            return title + " [SEP] " + body
        return title or body

arts["embed_text"] = arts.apply(prepare_text, axis=1)

# Statistik panjang teks (dalam kata)
word_counts = arts["embed_text"].str.split().str.len()
print(f"Panjang teks (kata): mean={word_counts.mean():.0f}, "
      f"median={word_counts.median():.0f}, max={word_counts.max():.0f}")
print(f"Artikel dengan >400 kata: {(word_counts > 400).sum()} ({(word_counts > 400).mean():.1%})")

## 4. Load model transformer

**FinBERT** (`ProsusAI/finbert`) adalah model BERT yang di-pre-train ulang pada Financial PhraseBank dan corpus keuangan lainnya. Model ini lebih memahami terminologi keuangan ("hawkish", "dovish", "yield curve inversion", dll.) dibandingkan BERT vanilla.

Kita menggunakan **`AutoModel`** (bukan `AutoModelForSequenceClassification`) karena kita ingin mengambil hidden states, bukan label sentimen.

In [ ]:
from transformers import AutoTokenizer, AutoModel

print(f"Loading model: {MODEL_NAME} ...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME)
model = model.to(DEVICE)
model.eval()  # mode inference, matikan dropout

EMBED_DIM = model.config.hidden_size
print(f"Model loaded! Embedding dim: {EMBED_DIM}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

if torch.cuda.is_available():
    mem_used = torch.cuda.memory_allocated() / 1024**3
    print(f"GPU memory used by model: {mem_used:.2f} GB")

## 5. Ekstraksi embedding per artikel

### Strategi Pooling: Mean Pooling

Dari hidden state layer terakhir, kita lakukan **mean pooling** atas semua token non-padding. Alasannya:

- `[CLS]` token embedding sering kurang representatif untuk teks panjang jika model tidak di-fine-tune dengan objective sentence-level.
- Mean pooling menghasilkan representasi yang lebih stabil dan telah terbukti bekerja baik di banyak benchmark (Reimers & Gurevych, 2019).

### Penanganan Teks Panjang

BERT-family memiliki batas 512 token. Artikel yang lebih panjang akan di-truncate. Ini bukan masalah besar karena:
- Paragraf awal biasanya berisi informasi paling penting ("inverted pyramid" dalam jurnalisme)
- Judul sudah digabung di awal, jadi pasti ter-encode

### Memory Management

Inference dilakukan dalam batch kecil (`BATCH_SIZE=8`) dengan `torch.no_grad()` dan embedding langsung dipindah ke CPU untuk menghemat VRAM.

In [ ]:
def mean_pooling(last_hidden_state, attention_mask):
    """
    Mean pooling: rata-rata hidden states di posisi non-padding.
    
    Args:
        last_hidden_state: (batch, seq_len, hidden_dim)
        attention_mask:    (batch, seq_len)
    Returns:
        embeddings: (batch, hidden_dim)
    """
    # Expand mask agar sesuai dimensi hidden state
    mask_expanded = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    # Jumlah hidden states yang tidak di-mask
    sum_embeddings = torch.sum(last_hidden_state * mask_expanded, dim=1)
    # Jumlah token non-padding (agar rata-rata, bukan jumlah)
    sum_mask = torch.clamp(mask_expanded.sum(dim=1), min=1e-9)
    return sum_embeddings / sum_mask


def extract_embeddings(texts, batch_size=BATCH_SIZE):
    """
    Ekstraksi embedding untuk list teks menggunakan mean pooling.
    
    Returns:
        numpy array shape (len(texts), hidden_dim)
    """
    all_embeddings = []
    
    for i in tqdm(range(0, len(texts), batch_size), desc="Embedding"):
        batch_texts = texts[i : i + batch_size]
        
        # Tokenisasi dengan padding dan truncation
        encoded = tokenizer(
            batch_texts,
            padding=True,
            truncation=True,
            max_length=MAX_LENGTH,
            return_tensors="pt",
        )
        encoded = {k: v.to(DEVICE) for k, v in encoded.items()}
        
        with torch.no_grad():
            outputs = model(**encoded)
        
        # Mean pooling dari last hidden state
        embeddings = mean_pooling(
            outputs.last_hidden_state, encoded["attention_mask"]
        )
        
        # Pindah ke CPU dan simpan sebagai numpy
        all_embeddings.append(embeddings.cpu().numpy())
        
        # Bersihkan GPU memory
        del encoded, outputs, embeddings
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    
    return np.vstack(all_embeddings)


print(f"Mengekstraksi embedding untuk {len(arts):,} artikel...")
print(f"Estimasi: {len(arts) // BATCH_SIZE + 1} batch × ~{BATCH_SIZE} artikel")

t0 = time.perf_counter()
article_embeddings = extract_embeddings(arts["embed_text"].tolist())
elapsed = time.perf_counter() - t0

print(f"\nSelesai dalam {elapsed:.1f} detik ({elapsed/60:.1f} menit)")
print(f"Shape embedding artikel: {article_embeddings.shape}")
print(f"Kecepatan: {len(arts)/elapsed:.1f} artikel/detik")

In [ ]:
# Simpan embedding mentah per artikel (untuk analisis lanjutan, misalnya t-SNE)
np.savez_compressed(
    FEATURE_DIR / f"embeddings_{MODEL_SHORT}_article.npz",
    embeddings=article_embeddings,
    urls=arts["url"].values,
    trading_dates=arts["trading_date"].astype(str).values,
    splits=arts["split"].values,
    model_name=MODEL_NAME,
)
print(f"Embedding per artikel tersimpan: embeddings_{MODEL_SHORT}_article.npz")
print(f"Ukuran file: {(FEATURE_DIR / f'embeddings_{MODEL_SHORT}_article.npz').stat().st_size / 1024**2:.1f} MB")

## 6. Bebaskan GPU memory

Setelah semua embedding diekstrak, model transformer tidak dibutuhkan lagi.
Kita hapus dari memory agar PCA dan langkah berikutnya tidak kehabisan RAM.

In [ ]:
del model, tokenizer
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory setelah cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GB")
print("Model transformer telah dibebaskan dari memory.")

## 7. Agregasi ke level hari bursa

Sama seperti notebook 03: satu hari bisa punya banyak artikel, sedangkan kurs hanya satu angka. Embedding semua artikel di hari yang sama dirata-ratakan.

Hari tanpa berita diisi vektor nol dan diberi penanda `no_news = 1`.

### Timing berita

Mengikuti setting yang sama dengan notebook 03 (`NEWS_TIMING`).

In [ ]:
# Tambahkan embedding ke dataframe artikel
emb_cols_raw = [f"emb_{i}" for i in range(article_embeddings.shape[1])]
emb_df = pd.DataFrame(article_embeddings, columns=emb_cols_raw)
arts_emb = pd.concat([arts.reset_index(drop=True), emb_df], axis=1)

# Agregasi harian: rata-rata embedding
daily_emb_agg = arts_emb.groupby("trading_date")[emb_cols_raw].mean()

# Merge dengan daily kurs (sama seperti notebook 03)
days = daily.copy()
if NEWS_TIMING == "sebelum_penetapan_besok":
    days["news_date"] = days["date"].shift(-1)
    keep = days["split"] == days["split"].shift(-1)
    days = days[keep].reset_index(drop=True)
else:
    days["news_date"] = days["date"]

days = days.merge(daily_emb_agg, left_on="news_date", right_index=True, how="left")

# Hari tanpa berita → vektor nol
has_news = days[emb_cols_raw[0]].notna()
days[emb_cols_raw] = days[emb_cols_raw].fillna(0.0)

print(f"Jumlah hari bursa: {len(days)}")
print(f"Hari dengan berita: {has_news.sum()} ({has_news.mean():.1%})")
print(f"Hari tanpa berita : {(~has_news).sum()} ({(~has_news).mean():.1%})")

## 8. Reduksi dimensi dengan PCA

Embedding FinBERT berdimensi 768, terlalu besar untuk ~1.000 baris data. Sama seperti TF-IDF → SVD di notebook 03, kita gunakan **PCA** untuk mereduksi ke `PCA_COMPONENTS` komponen.

- **Fit PCA hanya pada data train** → tidak ada leakage
- **Transform** ke semua data (train, val, test)
- PCA dipilih (bukan SVD) karena input sudah dense dan berdimensi sedang

In [ ]:
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

train_mask = (days["split"] == "train").values

# Ambil embedding harian
E = days[emb_cols_raw].values  # (n_days, hidden_dim)

# Standardisasi sebelum PCA (fit pada train)
scaler = StandardScaler()
E_scaled = scaler.fit_transform(E[train_mask])
E_all_scaled = scaler.transform(E)

# PCA fit pada train
n_comp = min(PCA_COMPONENTS, E_scaled.shape[1], E_scaled.shape[0] - 1)
pca = PCA(n_components=n_comp, random_state=42)
pca.fit(E_scaled)

# Transform semua
Z = pca.transform(E_all_scaled)
pca_cols = [f"transformer_{i+1}" for i in range(n_comp)]
pca_df = pd.DataFrame(Z, columns=pca_cols)

explained = pca.explained_variance_ratio_.sum()
print(f"{n_comp} komponen PCA menjelaskan {explained:.1%} variansi (train)")
print(f"Top-5 explained variance ratio: {pca.explained_variance_ratio_[:5].round(4)}")

In [ ]:
# Visualisasi cumulative explained variance
import matplotlib.pyplot as plt

cumvar = np.cumsum(pca.explained_variance_ratio_)
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, len(cumvar) + 1), cumvar, marker="o", markersize=3)
ax.axhline(0.9, ls="--", color="red", alpha=0.6, label="90% variansi")
ax.axhline(0.95, ls="--", color="orange", alpha=0.6, label="95% variansi")
ax.set_xlabel("Jumlah komponen PCA")
ax.set_ylabel("Cumulative explained variance")
ax.set_title(f"PCA pada embedding {MODEL_NAME.split('/')[-1]}")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(FEATURE_DIR / f"pca_variance_{MODEL_SHORT}.png", dpi=150)
plt.show()

# Berapa komponen untuk 90% dan 95%
for threshold in [0.8, 0.9, 0.95]:
    n_for_thresh = np.searchsorted(cumvar, threshold) + 1
    print(f"{threshold:.0%} variansi tercapai dengan {n_for_thresh} komponen")

## 9. Gabungkan menjadi dataset modeling

Dataset akhir berisi:
- Kolom identifikasi: `date`, `split`
- Target: `target` (3-class), `kurs_besok` (regresi)
- Fitur kurs historis: `ret_lag1..3`, `ret_std5`
- Fitur lexicon (dari notebook 03): VADER + Loughran-McDonald
- **Fitur transformer baru**: `transformer_1..50` (PCA dari FinBERT embedding)

In [ ]:
# Muat fitur lexicon dari notebook 03
lex = pd.read_csv(FEATURE_DIR / "features_lexicon.csv", parse_dates=["date"])

# Gabungkan: days (kurs + metadata) + PCA transformer embedding
model_df = pd.concat([days.reset_index(drop=True), pca_df], axis=1)

# Pastikan kolom yang dibutuhkan ada
essential_cols = ["date", "kurs_jisdor", "pct_change", "target", "kurs_besok",
                  "log_return", "ret_lag1", "ret_lag2", "ret_lag3", "ret_std5",
                  "delta_besok", "split"]

# Jika delta_besok belum ada, hitung
if "delta_besok" not in model_df.columns:
    model_df["delta_besok"] = model_df["kurs_besok"] - model_df["kurs_jisdor"]

# Merge fitur lexicon (VADER, LM) berdasarkan date
lex_feat_cols = [c for c in lex.columns if c not in ["date", "split"]]
model_df = model_df.merge(lex[["date"] + lex_feat_cols], on="date", how="left",
                          suffixes=("", "_lex"))

# Drop kolom duplikat jika ada
dup_cols = [c for c in model_df.columns if c.endswith("_lex")]
if dup_cols:
    for c in dup_cols:
        orig = c.replace("_lex", "")
        if orig in model_df.columns:
            model_df[orig] = model_df[orig].fillna(model_df[c])
    model_df = model_df.drop(columns=dup_cols)

# Simpan
out_file = FEATURE_DIR / "model_dataset_transformer.csv"
model_df.to_csv(out_file, index=False)
print(f"Dataset tersimpan: {out_file}")
print(f"Shape: {model_df.shape}")
print(f"Split: {model_df['split'].value_counts().to_dict()}")
print(f"\nKolom transformer: {pca_cols[:5]} ... {pca_cols[-1]}")
print(f"Kolom lexicon    : {[c for c in lex_feat_cols if c in model_df.columns][:5]} ...")
model_df.head()

## 10. Simpan juga embedding harian (sebelum PCA) untuk analisis lanjutan

In [ ]:
# Simpan embedding harian setelah PCA
daily_emb_out = days[["date", "split"]].copy()
daily_emb_out = pd.concat([daily_emb_out.reset_index(drop=True), pca_df], axis=1)
daily_emb_out.to_csv(FEATURE_DIR / f"embeddings_{MODEL_SHORT}_daily.csv", index=False)
print(f"Embedding harian (PCA) tersimpan: embeddings_{MODEL_SHORT}_daily.csv")

## 11. Analisis eksplorasi embedding

### 11a. Korelasi Spearman fitur transformer vs delta_besok (train)

Sama seperti cek di notebook 03 untuk fitur lexicon. Korelasi kemungkinan kecil karena kurs harian hampir random walk, tapi arah korelasi bisa informatif.

In [ ]:
tr = model_df[model_df["split"] == "train"]

# Korelasi Spearman transformer features vs delta_besok
corr_transformer = tr[pca_cols + ["delta_besok"]].corr(method="spearman")["delta_besok"].drop("delta_besok")
corr_sorted = corr_transformer.reindex(corr_transformer.abs().sort_values(ascending=False).index)

print("Top-10 komponen transformer yang paling berkorelasi dengan delta_besok (Spearman):")
print(corr_sorted.head(10).round(3).to_string())

# Bandingkan dengan korelasi lexicon features
lex_in_model = [c for c in lex_feat_cols if c in model_df.columns and c != "no_news"]
if lex_in_model:
    corr_lexicon = tr[lex_in_model + ["delta_besok"]].corr(method="spearman")["delta_besok"].drop("delta_besok")
    print("\nKorelasi lexicon features (pembanding):")
    print(corr_lexicon.sort_values().round(3).to_string())

### 11b. Visualisasi t-SNE embedding harian

Memetakan embedding transformer harian ke 2D menggunakan t-SNE.
Titik diwarnai berdasarkan label target (NAIK/STABIL/TURUN) untuk melihat apakah embedding mengelompokkan hari berdasarkan arah kurs.

In [ ]:
from sklearn.manifold import TSNE

# Gunakan data train saja untuk t-SNE (menghindari bias)
train_df = model_df[model_df["split"] == "train"].reset_index(drop=True)
X_tsne_input = train_df[pca_cols].values

tsne = TSNE(n_components=2, perplexity=30, random_state=42, n_iter=1000)
X_2d = tsne.fit_transform(X_tsne_input)

# Plot
colors_map = {"NAIK": "#2ecc71", "STABIL": "#3498db", "TURUN": "#e74c3c"}
fig, ax = plt.subplots(figsize=(8, 6))

for label, color in colors_map.items():
    mask = train_df["target"] == label
    ax.scatter(X_2d[mask, 0], X_2d[mask, 1], c=color, label=label,
              s=15, alpha=0.5, edgecolors="none")

ax.set_title(f"t-SNE: Embedding {MODEL_NAME.split('/')[-1]} harian (Train)")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(title="Target")
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FEATURE_DIR / f"tsne_{MODEL_SHORT}_daily.png", dpi=150)
plt.show()

print("Catatan: Jika titik-titik tidak terpisah jelas, itu wajar. Kurs harian mendekati random walk,")
print("sehingga pola semantik di berita sulit membentuk kluster yang jelas berdasarkan arah kurs.")

### 11c. Volatilitas tinggi vs rendah

Membandingkan embedding pada hari volatilitas tinggi vs rendah (berdasarkan `ret_std5`).
Apakah embedding menangkap perbedaan regime volatilitas?

In [ ]:
# Definisikan volatilitas tinggi/rendah berdasarkan quartile ret_std5
q75 = train_df["ret_std5"].quantile(0.75)
q25 = train_df["ret_std5"].quantile(0.25)

high_vol = train_df["ret_std5"] >= q75
low_vol  = train_df["ret_std5"] <= q25

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X_2d[high_vol, 0], X_2d[high_vol, 1], c="#e74c3c", label=f"Volatilitas tinggi (Q4, ≥{q75:.4f})",
          s=15, alpha=0.5, edgecolors="none")
ax.scatter(X_2d[low_vol, 0], X_2d[low_vol, 1], c="#2ecc71", label=f"Volatilitas rendah (Q1, ≤{q25:.4f})",
          s=15, alpha=0.5, edgecolors="none")
ax.scatter(X_2d[~high_vol & ~low_vol, 0], X_2d[~high_vol & ~low_vol, 1],
          c="#bdc3c7", label="Medium", s=8, alpha=0.3, edgecolors="none")

ax.set_title(f"t-SNE: Embedding {MODEL_NAME.split('/')[-1]} — Volatilitas Tinggi vs Rendah")
ax.set_xlabel("t-SNE 1")
ax.set_ylabel("t-SNE 2")
ax.legend(title="Volatilitas", fontsize=9)
ax.grid(alpha=0.2)
plt.tight_layout()
plt.savefig(FEATURE_DIR / f"tsne_{MODEL_SHORT}_volatility.png", dpi=150)
plt.show()

## 12. Ringkasan

Output yang dihasilkan notebook ini:

| File | Deskripsi |
|---|---|
| `embeddings_{model}_article.npz` | Embedding mentah per artikel (untuk analisis lanjutan) |
| `embeddings_{model}_daily.csv` | Embedding harian setelah PCA |
| `model_dataset_transformer.csv` | Dataset siap modeling (kurs + lag + lexicon + transformer) |
| `pca_variance_{model}.png` | Plot cumulative explained variance PCA |
| `tsne_{model}_daily.png` | Visualisasi t-SNE embedding harian |
| `tsne_{model}_volatility.png` | t-SNE: volatilitas tinggi vs rendah |

### Langkah selanjutnya

1. **Notebook 04b**: Modeling dengan fitur transformer → Model C (studi ablasi)
2. Bandingkan:
   - **Model A**: Hanya time-series
   - **Model B**: Time-series + NLP dasar (VADER, LM, TF-IDF) ← notebook 03 + 04
   - **Model C**: Time-series + fitur transformer ← notebook ini + 04b
3. Eksperimen tambahan: ganti `MODEL_NAME` ke model lain, bandingkan `EMBED_FIELD` title vs body

In [ ]:
# Metadata eksperimen
meta = {
    "model_name": MODEL_NAME,
    "model_short": MODEL_SHORT,
    "embed_dim": int(EMBED_DIM),
    "max_length": MAX_LENGTH,
    "batch_size": BATCH_SIZE,
    "embed_field": EMBED_FIELD,
    "pca_components": n_comp,
    "pca_explained_variance": round(float(explained), 4),
    "news_timing": NEWS_TIMING,
    "n_articles": len(arts),
    "n_days": len(days),
    "extraction_time_seconds": round(elapsed, 1),
    "dataset_shape": list(model_df.shape),
    "splits": model_df["split"].value_counts().to_dict(),
    "outputs": [
        f"embeddings_{MODEL_SHORT}_article.npz",
        f"embeddings_{MODEL_SHORT}_daily.csv",
        "model_dataset_transformer.csv",
        f"pca_variance_{MODEL_SHORT}.png",
        f"tsne_{MODEL_SHORT}_daily.png",
        f"tsne_{MODEL_SHORT}_volatility.png",
    ],
}

with open(FEATURE_DIR / f"transformer_extraction_meta_{MODEL_SHORT}.json", "w") as f:
    json.dump(meta, f, indent=2)

print("=" * 60)
print("SELESAI: Ekstraksi Fitur Contextual/Transformer Embeddings")
print("=" * 60)
for k, v in meta.items():
    print(f"  {k:30s}: {v}")